# Advanced Python Context Managers — Problems with Full Solutions

This notebook contains **advanced, self-contained practice problems** for Python context managers.

The problems build on these ideas:

- `try` / `finally` guarantees cleanup
- `with` statements automate setup and teardown
- custom context managers implement `__enter__` and `__exit__`
- `__exit__(exc_type, exc_value, traceback)` controls exception behavior
- context managers can solve patterns like **open/close**, **lock/release**, **change/reset**, **start/stop**, and **commit/rollback**

Each problem includes:

1. a problem statement,
2. a complete solution,
3. runnable examples,
4. assertion-based checks.

## Notebook Setup

Run this cell first. It imports only standard-library modules so the notebook remains portable.

In [1]:
from __future__ import annotations

import asyncio
import contextlib
import contextvars
import decimal
import io
import os
import random
import sqlite3
import sys
import tempfile
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, Iterator, Optional

## Best Practices Checklist

Use this checklist while solving the problems.

- Put cleanup code in `__exit__` or after `yield` in a generator-based context manager.
- Return `False` or `None` from `__exit__` unless you intentionally want to suppress an exception.
- Suppress only specific exception types; avoid hiding unexpected failures.
- Make cleanup idempotent when possible, meaning repeated cleanup should not break.
- Prefer `contextlib.contextmanager` for simple setup/teardown.
- Prefer a class-based context manager when you need reusable state, multiple methods, or richer behavior.
- Prefer `contextlib.ExitStack` when the number of managed resources is dynamic.
- Write small tests that prove cleanup still happens when exceptions occur.

## Problem 1 — Build a Safe Temporary File Writer

Create a class-based context manager called `AtomicTextWriter`.

Requirements:

- It receives a target file path.
- Inside the `with` block, writes go to a temporary file.
- If the block exits successfully, replace the target file atomically.
- If the block exits with an exception, delete the temporary file and leave the original file unchanged.
- Return the opened temporary file from `__enter__`.

In [2]:
class AtomicTextWriter:
    # Atomically write text to a target path.
    #
    # On success:
    #     temp file -> target path
    #
    # On failure:
    #     temp file is deleted
    #     target path remains unchanged

    def __init__(self, target: str | os.PathLike[str], encoding: str = "utf-8"):
        self.target = Path(target)
        self.encoding = encoding
        self._temp_file = None
        self._temp_path: Optional[Path] = None

    def __enter__(self):
        self.target.parent.mkdir(parents=True, exist_ok=True)

        fd, temp_name = tempfile.mkstemp(
            prefix=f".{self.target.name}.",
            suffix=".tmp",
            dir=str(self.target.parent),
            text=True,
        )

        self._temp_path = Path(temp_name)
        self._temp_file = os.fdopen(fd, mode="w", encoding=self.encoding)
        return self._temp_file

    def __exit__(self, exc_type, exc_value, traceback):
        if self._temp_file is not None and not self._temp_file.closed:
            self._temp_file.close()

        if exc_type is None:
            os.replace(self._temp_path, self.target)
        else:
            if self._temp_path is not None:
                self._temp_path.unlink(missing_ok=True)

        return False


with tempfile.TemporaryDirectory() as tmp:
    target = Path(tmp) / "data.txt"
    target.write_text("original", encoding="utf-8")

    with AtomicTextWriter(target) as f:
        f.write("new content")

    assert target.read_text(encoding="utf-8") == "new content"

    try:
        with AtomicTextWriter(target) as f:
            f.write("broken content")
            raise RuntimeError("simulate failure")
    except RuntimeError:
        pass

    assert target.read_text(encoding="utf-8") == "new content"

print("Problem 1 passed")

Problem 1 passed


## Problem 2 — Measure Runtime with a Class-Based Timer

Create a context manager called `Timer`.

Requirements:

- Use `time.perf_counter()`.
- Record `start`, `end`, and `elapsed`.
- Return the timer object from `__enter__`.
- Never suppress exceptions.
- Store optional labels for readable output.

In [3]:
class Timer:
    def __init__(self, label: str = "timer"):
        self.label = label
        self.start: Optional[float] = None
        self.end: Optional[float] = None
        self.elapsed: Optional[float] = None

    def __enter__(self) -> "Timer":
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.end = time.perf_counter()
        self.elapsed = self.end - self.start
        return False


with Timer("small calculation") as timer:
    total = sum(range(100_000))

assert total == 4_999_950_000
assert timer.start is not None
assert timer.end is not None
assert timer.elapsed is not None
assert timer.elapsed >= 0

print(f"{timer.label}: {timer.elapsed:.6f} seconds")
print("Problem 2 passed")

small calculation: 0.003674 seconds
Problem 2 passed


## Problem 3 — Selective Exception Suppression

Create `SuppressOnly`.

Requirements:

- Accept one or more exception classes.
- Suppress only those exceptions.
- Record the suppressed exception object in `self.exception`.
- Let all other exceptions propagate.

In [4]:
class SuppressOnly:
    def __init__(self, *exception_types: type[BaseException]):
        if not exception_types:
            raise ValueError("Pass at least one exception type.")
        self.exception_types = exception_types
        self.exception: Optional[BaseException] = None

    def __enter__(self) -> "SuppressOnly":
        self.exception = None
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            return False

        if issubclass(exc_type, self.exception_types):
            self.exception = exc_value
            return True

        return False


with SuppressOnly(KeyError) as ctx:
    raise KeyError("missing key")

assert isinstance(ctx.exception, KeyError)

try:
    with SuppressOnly(KeyError):
        raise ValueError("wrong error")
except ValueError:
    propagated = True
else:
    propagated = False

assert propagated is True

print("Problem 3 passed")

Problem 3 passed


## Problem 4 — Temporarily Change Environment Variables

Create `temporary_env`.

Requirements:

- Temporarily set environment variables.
- Restore previous values when the block exits.
- If a variable did not exist before, remove it during cleanup.
- Cleanup must happen even if an exception occurs.

In [5]:
class temporary_env:
    def __init__(self, **updates: str):
        self.updates = updates
        self._old_values: dict[str, Optional[str]] = {}

    def __enter__(self) -> "temporary_env":
        for key, value in self.updates.items():
            self._old_values[key] = os.environ.get(key)
            os.environ[key] = value
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        for key, old_value in self._old_values.items():
            if old_value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = old_value
        return False


os.environ.pop("APP_MODE", None)

with temporary_env(APP_MODE="test"):
    assert os.environ["APP_MODE"] == "test"

assert "APP_MODE" not in os.environ

os.environ["APP_MODE"] = "prod"

try:
    with temporary_env(APP_MODE="debug"):
        assert os.environ["APP_MODE"] == "debug"
        raise RuntimeError("fail inside block")
except RuntimeError:
    pass

assert os.environ["APP_MODE"] == "prod"

print("Problem 4 passed")

Problem 4 passed


## Problem 5 — Capture Printed Output

Create a context manager called `CaptureStdout`.

Requirements:

- Temporarily replace `sys.stdout` with an in-memory text buffer.
- Return the buffer from `__enter__`.
- Restore the original `sys.stdout` in `__exit__`.
- Prove that restoration happens after an exception.

In [6]:
class CaptureStdout:
    def __init__(self):
        self._old_stdout = None
        self.buffer = io.StringIO()

    def __enter__(self) -> io.StringIO:
        self._old_stdout = sys.stdout
        sys.stdout = self.buffer
        return self.buffer

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        sys.stdout = self._old_stdout
        return False


with CaptureStdout() as captured:
    print("hello")
    print("world")

assert captured.getvalue() == "hello\nworld\n"

original_stdout = sys.stdout

try:
    with CaptureStdout() as captured:
        print("before error")
        raise ValueError("boom")
except ValueError:
    pass

assert sys.stdout is original_stdout
assert captured.getvalue() == "before error\n"

print("Problem 5 passed")

Problem 5 passed


## Problem 6 — Use `contextlib.contextmanager` for Temporary Decimal Precision

Implement `decimal_precision(precision)` using `@contextlib.contextmanager`.

Requirements:

- Temporarily change the active decimal precision.
- Restore the old precision after the block.
- Work correctly when exceptions occur.

In [7]:
@contextlib.contextmanager
def decimal_precision(precision: int) -> Iterator[decimal.Context]:
    old_context = decimal.getcontext().copy()
    decimal.getcontext().prec = precision

    try:
        yield decimal.getcontext()
    finally:
        decimal.setcontext(old_context)


original_precision = decimal.getcontext().prec

with decimal_precision(4):
    result = decimal.Decimal("1") / decimal.Decimal("7")
    assert str(result) == "0.1429"

assert decimal.getcontext().prec == original_precision

try:
    with decimal_precision(3):
        assert decimal.getcontext().prec == 3
        raise RuntimeError("precision block failed")
except RuntimeError:
    pass

assert decimal.getcontext().prec == original_precision

print("Problem 6 passed")

Problem 6 passed


## Problem 7 — Transactional List Updates

Create a context manager called `ListTransaction`.

Requirements:

- Accept a list.
- Inside the block, allow normal mutation of the list.
- If the block exits successfully, keep the changes.
- If the block exits with an exception, restore the list to its original contents.
- Return the original list object, not a copy.

In [8]:
class ListTransaction:
    def __init__(self, items: list[Any]):
        self.items = items
        self._snapshot: list[Any] = []

    def __enter__(self) -> list[Any]:
        self._snapshot = self.items.copy()
        return self.items

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is not None:
            self.items[:] = self._snapshot
        return False


numbers = [1, 2, 3]

with ListTransaction(numbers) as tx:
    tx.append(4)
    tx[0] = 100

assert numbers == [100, 2, 3, 4]

try:
    with ListTransaction(numbers) as tx:
        tx.append(999)
        tx.clear()
        raise RuntimeError("rollback")
except RuntimeError:
    pass

assert numbers == [100, 2, 3, 4]

print("Problem 7 passed")

Problem 7 passed


## Problem 8 — Commit/Rollback with SQLite

Create `SQLiteTransaction`.

Requirements:

- Accept a SQLite connection.
- On successful exit, commit.
- On failed exit, rollback.
- Do not suppress exceptions.
- Demonstrate both commit and rollback.

In [9]:
class SQLiteTransaction:
    def __init__(self, connection: sqlite3.Connection):
        self.connection = connection

    def __enter__(self) -> sqlite3.Connection:
        return self.connection

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is None:
            self.connection.commit()
        else:
            self.connection.rollback()
        return False


conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT)")

with SQLiteTransaction(conn) as db:
    db.execute("INSERT INTO users (name) VALUES (?)", ("Ada",))

rows = conn.execute("SELECT name FROM users").fetchall()
assert rows == [("Ada",)]

try:
    with SQLiteTransaction(conn) as db:
        db.execute("INSERT INTO users (name) VALUES (?)", ("Grace",))
        raise RuntimeError("cancel insert")
except RuntimeError:
    pass

rows = conn.execute("SELECT name FROM users").fetchall()
assert rows == [("Ada",)]

conn.close()

print("Problem 8 passed")

Problem 8 passed


## Problem 9 — Dynamic Resource Management with `ExitStack`

Create a function `read_all(paths)`.

Requirements:

- Accept a list of file paths.
- Open all files safely.
- Use `contextlib.ExitStack` because the number of files is dynamic.
- Return a list of file contents.
- Prove that all files are closed after leaving the stack.

In [10]:
def read_all(paths: Iterable[Path]) -> tuple[list[str], list[Any]]:
    opened_files = []

    with contextlib.ExitStack() as stack:
        for path in paths:
            f = stack.enter_context(open(path, "r", encoding="utf-8"))
            opened_files.append(f)

        contents = [f.read() for f in opened_files]

    return contents, opened_files


with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    paths = []

    for index, text in enumerate(["alpha", "beta", "gamma"], start=1):
        path = base / f"file_{index}.txt"
        path.write_text(text, encoding="utf-8")
        paths.append(path)

    contents, handles = read_all(paths)

assert contents == ["alpha", "beta", "gamma"]
assert all(handle.closed for handle in handles)

print("Problem 9 passed")

Problem 9 passed


## Problem 10 — Register Cleanup Callbacks with `ExitStack`

Use `contextlib.ExitStack` to build `ResourceGroup`.

Requirements:

- Allow arbitrary cleanup callbacks to be registered.
- Execute callbacks in reverse order.
- Run all callbacks even if the block raises.
- Return the resource group itself from `__enter__`.

In [11]:
class ResourceGroup:
    def __init__(self):
        self._stack = contextlib.ExitStack()

    def __enter__(self) -> "ResourceGroup":
        self._stack.__enter__()
        return self

    def callback(self, func: Callable[..., Any], *args, **kwargs) -> None:
        self._stack.callback(func, *args, **kwargs)

    def enter_context(self, cm):
        return self._stack.enter_context(cm)

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        return self._stack.__exit__(exc_type, exc_value, traceback)


events = []

try:
    with ResourceGroup() as group:
        group.callback(events.append, "cleanup 1")
        group.callback(events.append, "cleanup 2")
        events.append("inside")
        raise RuntimeError("boom")
except RuntimeError:
    pass

assert events == ["inside", "cleanup 2", "cleanup 1"]

print("Problem 10 passed")

Problem 10 passed


## Problem 11 — Lock / Release Pattern

Create `locked`.

Requirements:

- Accept a `threading.Lock` or `threading.RLock`.
- Acquire the lock in `__enter__`.
- Release the lock in `__exit__`.
- Return the lock object.
- Prove the lock is released after an exception.

In [12]:
class locked:
    def __init__(self, lock):
        self.lock = lock

    def __enter__(self):
        self.lock.acquire()
        return self.lock

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.lock.release()
        return False


lock = threading.RLock()

with locked(lock):
    # RLock lets the same thread acquire again.
    assert lock.acquire(blocking=False) is True
    lock.release()

try:
    with locked(lock):
        raise RuntimeError("failure while locked")
except RuntimeError:
    pass

assert lock.acquire(blocking=False) is True
lock.release()

print("Problem 11 passed")

Problem 11 passed


## Problem 12 — Temporary Random State

Create `random_state`.

Requirements:

- Save the current state of Python's global `random` generator.
- Seed it inside the block.
- Restore the old state afterward.
- The same seed should produce the same sequence.
- The outer random sequence should continue as if the temporary block never happened.

In [13]:
@contextlib.contextmanager
def random_state(seed: int) -> Iterator[None]:
    old_state = random.getstate()
    random.seed(seed)

    try:
        yield
    finally:
        random.setstate(old_state)


with random_state(42):
    seq1 = [random.randint(1, 100) for _ in range(5)]

with random_state(42):
    seq2 = [random.randint(1, 100) for _ in range(5)]

assert seq1 == seq2

random.seed(123)
first = random.random()

with random_state(999):
    _ = [random.random() for _ in range(10)]

second_after_context = random.random()

random.seed(123)
_ = random.random()
expected_second = random.random()

assert second_after_context == expected_second

print("Problem 12 passed")

Problem 12 passed


## Problem 13 — Change and Reset Object Attributes

Create `temporary_attributes`.

Requirements:

- Temporarily set attributes on any object.
- Restore old values on exit.
- Delete attributes that did not exist before.
- Work correctly if the block raises an exception.

In [14]:
_MISSING = object()

class temporary_attributes:
    def __init__(self, obj: Any, **updates: Any):
        self.obj = obj
        self.updates = updates
        self._old_values: dict[str, Any] = {}

    def __enter__(self) -> Any:
        for name, value in self.updates.items():
            self._old_values[name] = getattr(self.obj, name, _MISSING)
            setattr(self.obj, name, value)
        return self.obj

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        for name, old_value in self._old_values.items():
            if old_value is _MISSING:
                delattr(self.obj, name)
            else:
                setattr(self.obj, name, old_value)
        return False


@dataclass
class Config:
    debug: bool = False


config = Config()

with temporary_attributes(config, debug=True, api_url="http://localhost"):
    assert config.debug is True
    assert config.api_url == "http://localhost"

assert config.debug is False
assert not hasattr(config, "api_url")

try:
    with temporary_attributes(config, debug=True):
        raise RuntimeError("crash")
except RuntimeError:
    pass

assert config.debug is False

print("Problem 13 passed")

Problem 13 passed


## Problem 14 — Context Manager as a Decorator

Create `log_calls` using `contextlib.ContextDecorator`.

Requirements:

- Can be used both as a context manager and as a decorator.
- Append `"enter"` before the block/function.
- Append `"exit"` afterward.
- Never suppress exceptions.

In [15]:
class log_calls(contextlib.ContextDecorator):
    def __init__(self, events: list[str]):
        self.events = events

    def __enter__(self):
        self.events.append("enter")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.events.append("exit")
        return False


events = []

with log_calls(events):
    events.append("inside context")

assert events == ["enter", "inside context", "exit"]

decorator_events = []

@log_calls(decorator_events)
def add(a, b):
    decorator_events.append("inside function")
    return a + b

assert add(2, 3) == 5
assert decorator_events == ["enter", "inside function", "exit"]

print("Problem 14 passed")

Problem 14 passed


## Problem 15 — Build an HTML Tag Builder

Create `HTMLBuilder`.

Requirements:

- Use a context manager for nested tags.
- `with html.tag("ul"):` should open and close a tag.
- Support text nodes.
- Track correct nesting.
- Return the final HTML string.

In [16]:
class HTMLBuilder:
    def __init__(self):
        self._parts: list[str] = []

    @contextlib.contextmanager
    def tag(self, name: str, **attrs: str) -> Iterator[None]:
        attr_text = "".join(f' {key.rstrip("_")}="{value}"' for key, value in attrs.items())
        self._parts.append(f"<{name}{attr_text}>")

        try:
            yield
        finally:
            self._parts.append(f"</{name}>")

    def text(self, value: str) -> None:
        # Minimal escaping for common HTML-sensitive characters.
        escaped = (
            value.replace("&", "&amp;")
                 .replace("<", "&lt;")
                 .replace(">", "&gt;")
        )
        self._parts.append(escaped)

    def render(self) -> str:
        return "".join(self._parts)


html = HTMLBuilder()

with html.tag("ul", class_="items"):
    with html.tag("li"):
        html.text("Ada")
    with html.tag("li"):
        html.text("Grace <Hopper>")

result = html.render()

assert result == (
    '<ul class="items">'
    "<li>Ada</li>"
    "<li>Grace &lt;Hopper&gt;</li>"
    "</ul>"
)

print(result)
print("Problem 15 passed")

<ul class="items"><li>Ada</li><li>Grace &lt;Hopper&gt;</li></ul>
Problem 15 passed


## Problem 16 — Generator-Based Error Translation

Create `translate_errors`.

Requirements:

- Implement with `@contextlib.contextmanager`.
- Convert one exception type into another.
- Preserve the original exception as the cause using `raise ... from ...`.
- Do not change unrelated exception types.

In [17]:
@contextlib.contextmanager
def translate_errors(
    source_error: type[BaseException],
    target_error: type[BaseException],
    message: str,
) -> Iterator[None]:
    try:
        yield
    except source_error as exc:
        raise target_error(message) from exc


try:
    with translate_errors(KeyError, RuntimeError, "configuration key missing"):
        raise KeyError("API_TOKEN")
except RuntimeError as exc:
    translated = exc
else:
    translated = None

assert isinstance(translated, RuntimeError)
assert isinstance(translated.__cause__, KeyError)

try:
    with translate_errors(KeyError, RuntimeError, "configuration key missing"):
        raise ValueError("unrelated")
except ValueError:
    unrelated_propagated = True
else:
    unrelated_propagated = False

assert unrelated_propagated is True

print("Problem 16 passed")

Problem 16 passed


## Problem 17 — Async Context Manager

Create `AsyncConnection`.

Requirements:

- Use `async with`.
- Simulate connecting and disconnecting with `asyncio.sleep(0)`.
- Set `connected` to `True` in `__aenter__`.
- Set `connected` to `False` in `__aexit__`.
- Return the object itself.
- Never suppress exceptions.

In [18]:
class AsyncConnection:
    def __init__(self):
        self.connected = False
        self.events: list[str] = []

    async def __aenter__(self) -> "AsyncConnection":
        await asyncio.sleep(0)
        self.connected = True
        self.events.append("connect")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback) -> bool:
        await asyncio.sleep(0)
        self.connected = False
        self.events.append("disconnect")
        return False


async def demo_async_connection():
    connection = AsyncConnection()

    async with connection as conn:
        assert conn.connected is True
        conn.events.append("inside")

    assert connection.connected is False
    assert connection.events == ["connect", "inside", "disconnect"]

    try:
        async with connection:
            raise RuntimeError("async failure")
    except RuntimeError:
        pass

    assert connection.connected is False
    assert connection.events[-1] == "disconnect"

    return connection.events


events = await demo_async_connection()

print(events)
print("Problem 17 passed")

['connect', 'inside', 'disconnect', 'connect', 'disconnect']
Problem 17 passed


## Problem 18 — Async Generator Context Manager

Create `async_timer` using `@contextlib.asynccontextmanager`.

Requirements:

- Measure elapsed time in an async block.
- Yield a dictionary that is filled with elapsed time on exit.
- Work even if the async block raises.

In [19]:
@contextlib.asynccontextmanager
async def async_timer(label: str = "async timer"):
    info = {"label": label, "elapsed": None}
    start = time.perf_counter()

    try:
        yield info
    finally:
        info["elapsed"] = time.perf_counter() - start


async def demo_async_timer():
    async with async_timer("sleep zero") as info:
        await asyncio.sleep(0)

    assert info["label"] == "sleep zero"
    assert info["elapsed"] is not None
    assert info["elapsed"] >= 0

    try:
        async with async_timer("failing async block") as failed_info:
            await asyncio.sleep(0)
            raise RuntimeError("boom")
    except RuntimeError:
        pass

    assert failed_info["elapsed"] is not None
    return info, failed_info


info, failed_info = await demo_async_timer()

print(info)
print(failed_info)
print("Problem 18 passed")

{'label': 'sleep zero', 'elapsed': 0.0001038997434079647}
{'label': 'failing async block', 'elapsed': 5.4900068789720535e-05}
Problem 18 passed


## Problem 19 — Reentrant Context Manager

A context manager is **reentrant** if the same instance can be entered multiple times safely.

Create `ReentrantCounter`.

Requirements:

- The same object can be used in nested `with` blocks.
- Track current nesting depth.
- Track maximum depth reached.
- Return the context manager itself.
- Restore depth correctly even if an inner block raises.

In [20]:
class ReentrantCounter:
    def __init__(self):
        self.depth = 0
        self.max_depth = 0

    def __enter__(self) -> "ReentrantCounter":
        self.depth += 1
        self.max_depth = max(self.max_depth, self.depth)
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.depth -= 1
        return False


counter = ReentrantCounter()

with counter as c1:
    assert c1.depth == 1

    with counter as c2:
        assert c2 is c1
        assert counter.depth == 2

    assert counter.depth == 1

assert counter.depth == 0
assert counter.max_depth == 2

try:
    with counter:
        with counter:
            raise RuntimeError("nested failure")
except RuntimeError:
    pass

assert counter.depth == 0
assert counter.max_depth == 2

print("Problem 19 passed")

Problem 19 passed


## Problem 20 — Context Variables: Temporary Request ID

Use `contextvars.ContextVar` to manage request-scoped data.

Create `request_id`.

Requirements:

- Temporarily set a request ID.
- Restore the previous value using the token returned by `.set()`.
- Work correctly with nested contexts.

In [21]:
current_request_id = contextvars.ContextVar("current_request_id", default=None)

@contextlib.contextmanager
def request_id(value: str) -> Iterator[str]:
    token = current_request_id.set(value)

    try:
        yield value
    finally:
        current_request_id.reset(token)


assert current_request_id.get() is None

with request_id("outer"):
    assert current_request_id.get() == "outer"

    with request_id("inner"):
        assert current_request_id.get() == "inner"

    assert current_request_id.get() == "outer"

assert current_request_id.get() is None

print("Problem 20 passed")

Problem 20 passed


## Problem 21 — Safer Nested File Processing with `ExitStack` and Validation

Create `combine_files`.

Requirements:

- Accept input paths and an output path.
- Open an arbitrary number of input files.
- Open the output file only after all input files were successfully opened.
- Combine file contents with newline separators.
- If opening any input fails, do not create the output file.

In [22]:
def combine_files(input_paths: list[Path], output_path: Path) -> None:
    with contextlib.ExitStack() as stack:
        inputs = [
            stack.enter_context(open(path, "r", encoding="utf-8"))
            for path in input_paths
        ]

        output = stack.enter_context(open(output_path, "w", encoding="utf-8"))

        for index, f in enumerate(inputs):
            if index:
                output.write("\n")
            output.write(f.read())


with tempfile.TemporaryDirectory() as tmp:
    base = Path(tmp)
    first = base / "first.txt"
    second = base / "second.txt"
    missing = base / "missing.txt"
    output = base / "combined.txt"

    first.write_text("one", encoding="utf-8")
    second.write_text("two", encoding="utf-8")

    combine_files([first, second], output)
    assert output.read_text(encoding="utf-8") == "one\ntwo"

    output.unlink()

    try:
        combine_files([first, missing, second], output)
    except FileNotFoundError:
        pass

    assert not output.exists()

print("Problem 21 passed")

Problem 21 passed


## Problem 22 — Design a Context Manager with Explicit Commit

Create `ManualCommitTransaction`.

Requirements:

- Wrap a list.
- Inside the block, mutations happen directly on the list.
- By default, rollback changes when exiting.
- If `.commit()` is called, keep the changes.
- If an exception occurs, rollback even if `.commit()` was called.

In [23]:
class ManualCommitTransaction:
    def __init__(self, items: list[Any]):
        self.items = items
        self._snapshot: list[Any] = []
        self._committed = False

    def __enter__(self) -> "ManualCommitTransaction":
        self._snapshot = self.items.copy()
        self._committed = False
        return self

    def commit(self) -> None:
        self._committed = True

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        if exc_type is not None or not self._committed:
            self.items[:] = self._snapshot
        return False


items = ["a"]

with ManualCommitTransaction(items) as tx:
    tx.items.append("b")
    tx.commit()

assert items == ["a", "b"]

with ManualCommitTransaction(items) as tx:
    tx.items.append("c")
    # no commit

assert items == ["a", "b"]

try:
    with ManualCommitTransaction(items) as tx:
        tx.items.append("d")
        tx.commit()
        raise RuntimeError("failure wins over commit")
except RuntimeError:
    pass

assert items == ["a", "b"]

print("Problem 22 passed")

Problem 22 passed


## Problem 23 — Debug Exception Details in `__exit__`

Create `ExceptionInspector`.

Requirements:

- Store `exc_type`, `exc_value`, and `traceback` on the context manager.
- Do not suppress exceptions by default.
- Demonstrate that values are `None` when no exception occurs.
- Demonstrate that values are populated when an exception occurs.

In [24]:
class ExceptionInspector:
    def __init__(self, suppress: bool = False):
        self.suppress = suppress
        self.exc_type = None
        self.exc_value = None
        self.traceback = None

    def __enter__(self) -> "ExceptionInspector":
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        self.exc_type = exc_type
        self.exc_value = exc_value
        self.traceback = traceback
        return self.suppress


with ExceptionInspector() as clean:
    x = 10 + 5

assert clean.exc_type is None
assert clean.exc_value is None
assert clean.traceback is None

with ExceptionInspector(suppress=True) as failed:
    raise ValueError("example")

assert failed.exc_type is ValueError
assert isinstance(failed.exc_value, ValueError)
assert failed.traceback is not None

print("Problem 23 passed")

Problem 23 passed


## Problem 24 — Final Challenge: A Composite Test Sandbox

Create `TestSandbox`.

Requirements:

- Create a temporary directory.
- Temporarily change the working directory to that directory.
- Capture stdout.
- Provide helper method `.path(name)`.
- Clean up everything on exit.
- Restore working directory and stdout even if an exception occurs.

In [25]:
class TestSandbox:
    def __init__(self):
        self._tempdir_cm = tempfile.TemporaryDirectory()
        self._stdout_cm = CaptureStdout()
        self.old_cwd: Optional[Path] = None
        self.root: Optional[Path] = None
        self.stdout: Optional[io.StringIO] = None

    def __enter__(self) -> "TestSandbox":
        self.old_cwd = Path.cwd()
        self.root = Path(self._tempdir_cm.__enter__())

        os.chdir(self.root)
        self.stdout = self._stdout_cm.__enter__()

        return self

    def path(self, name: str) -> Path:
        if self.root is None:
            raise RuntimeError("Sandbox has not been entered.")
        return self.root / name

    def __exit__(self, exc_type, exc_value, traceback) -> bool:
        try:
            self._stdout_cm.__exit__(exc_type, exc_value, traceback)
        finally:
            if self.old_cwd is not None:
                os.chdir(self.old_cwd)
            self._tempdir_cm.__exit__(exc_type, exc_value, traceback)

        return False


original_cwd = Path.cwd()
original_stdout = sys.stdout

with TestSandbox() as sandbox:
    Path("example.txt").write_text("inside sandbox", encoding="utf-8")
    print("captured message")

    assert Path.cwd() == sandbox.root
    assert sandbox.path("example.txt").read_text(encoding="utf-8") == "inside sandbox"

assert Path.cwd() == original_cwd
assert sys.stdout is original_stdout
assert sandbox.stdout.getvalue() == "captured message\n"
assert not sandbox.root.exists()

try:
    with TestSandbox() as failing_sandbox:
        print("before failure")
        raise RuntimeError("sandbox error")
except RuntimeError:
    pass

assert Path.cwd() == original_cwd
assert sys.stdout is original_stdout
assert not failing_sandbox.root.exists()
assert failing_sandbox.stdout.getvalue() == "before failure\n"

print("Problem 24 passed")

Problem 24 passed


# Summary

You practiced advanced context-manager patterns:

- atomic file replacement
- timing
- selective exception suppression
- temporary environment and object changes
- output capture
- generator-based context managers
- commit/rollback transactions
- `ExitStack`
- lock management
- random state restoration
- decorators
- HTML building
- async context managers
- reentrancy
- context variables
- composite sandboxes

A strong mental model:

> A context manager is a disciplined way to guarantee that every setup has a matching cleanup.